# 04 ML Classical Algorithms

This notebook uses `MLTrainAndStore` to train regressors with only the classical-network features.

In [1]:
import sys
import os
from pathlib import Path

%matplotlib inline

sys.path.insert(0, os.path.abspath('../..'))
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    ModelTrainer,
    load_classical_dataset,
    make_pipeline,
)

pd.set_option("display.max_columns", 200)
PROJECT_ROOT = Path().resolve().parents[1]

## Load Dataset

In [2]:
print(PROJECT_ROOT)

C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


In [3]:
df, feature_cols = load_classical_dataset(PROJECT_ROOT)
df.shape, feature_cols

((145536, 89),
 ['degree_centrality_in',
  'degree_centrality_out',
  'degree_centrality_total',
  'betweenness_centrality',
  'closeness_centrality',
  'weighted_degree_in',
  'weighted_degree_out',
  'weighted_degree_total',
  'pagerank',
  'debtrank'])

In [4]:
trainer = ModelTrainer(
    df=df,
    feature_cols=feature_cols,
    target_col="log_systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 89), (18192, 89), (13644, 89))

## Define Models

In [5]:
candidate_models = {
    "linear_regression": make_pipeline(LinearRegression(), scale_features=True),
}

list(candidate_models)

['linear_regression']

## Train And Store

In [6]:
trainer.train_all(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.023841,0.033423,0.028335,0.126829,0.17679,0.143692,0.484254,0.439928,0.489121


In [7]:
trainer.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.023841,0.033423,0.028335,0.126829,0.17679,0.143692,0.484254,0.439928,0.489121


## Single-Model Pattern

Use this when you want to add one model manually.

In [8]:
amodel = make_pipeline(
    XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    scale_features=True,
)

trainer.train(model=amodel, name="XGBRegressor_search_1")
trainer.leaderboard()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,XGBRegressor_search_1,0.006841,0.013899,0.010625,0.040947,0.082431,0.063494,0.946242,0.878239,0.900248
1,linear_regression,0.023841,0.033423,0.028335,0.126829,0.17679,0.143692,0.484254,0.439928,0.489121


## Best Model

In [9]:
trainer.best_name()

'XGBRegressor_search_1'

In [10]:
trainer.test_predictions().head(20)

,bank_id,year,quarter,period,log_systemic_risk_label,prediction,abs_error
0,53,2023,2,2023Q2,2.397895,1.254792,1.143103
1,61,2023,1,2023Q1,0.693147,1.805626,1.112478
2,5,2023,1,2023Q1,4.290459,3.391862,0.898597
3,28,2023,1,2023Q1,3.178054,2.356483,0.821571
4,1,2023,1,2023Q1,3.737670,2.921669,0.816001
5,6,2023,1,2023Q1,3.761200,2.945373,0.815827
6,120,2023,3,2023Q3,1.791759,0.980666,0.811093
7,22,2023,2,2023Q2,1.791759,0.988526,0.803234
8,5,2023,2,2023Q2,4.110874,3.331663,0.779211
9,114,2023,3,2023Q3,0.693147,1.467633,0.774486
